# 06 — Robustness

## What this notebook does
This notebook evaluates the sensitivity of the baseline BVAR results from notebook 05 while changing one modelling choice at a time.

The checks reproduce the final Master's Thesis robustness analysis:
1. **Reduced system** — remove Brent and Gas and compare the shared macro-financial responses
2. **Structural identification and Gas-response robustness** — invert Brent/Gas, place energy before GPR, exclude the 2021–2022 gas-crisis window, and use a pre-2022Q3 sample
3. **Covariance-prior sensitivity** — compare the baseline `S=0` prior with `S=I` and the scale-aware `S=diag(Sigma_OLS)` alternative
4. **Levels versus differences** — re-estimate the 16 systems in log levels and compare signs, detectability and geographical heterogeneity; Johansen diagnostics provide supporting evidence

All checks retain the baseline lag order, Minnesota hyperparameters, Gibbs settings and frozen COVID-volatility parameters unless the robustness exercise explicitly changes that element.


## A. Setup and baseline results

In [1]:
# ============================================================
# A. SETUP AND BASELINE RESULTS
# ============================================================
from pathlib import Path
from itertools import product
import pickle
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# Support both Google Colab and local execution
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/TFM_GitHub")
except ImportError:
    PROJECT_ROOT = Path.cwd().resolve()
    if PROJECT_ROOT.name == "notebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "notebooks").exists():
    raise FileNotFoundError(
        f"Could not find the project under: {PROJECT_ROOT}"
    )

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
COVID_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "04_covid_volatility"
BASELINE_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "05_bvar_main_results"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "06_robustness"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
for directory in [OUTPUT_DIR, TABLE_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

BVAR_READY_PATH = PROCESSED_DIR / "03_bvar_ready_panel.xlsx"
CLEAN_PANEL_PATH = PROCESSED_DIR / "03_clean_quarterly_panel.xlsx"
COVID_THETA_PATH = COVID_OUTPUT_DIR / "PRODUCTION_covid_theta_mle.csv"
BASELINE_RESULTS_PATH = BASELINE_OUTPUT_DIR / "bvar_results_main.pkl"

for path in [BVAR_READY_PATH, CLEAN_PANEL_PATH, COVID_THETA_PATH, BASELINE_RESULTS_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Required input not found: {path}")

panel = pd.read_excel(BVAR_READY_PATH)
panel["quarter"] = pd.PeriodIndex(panel["quarter"].astype(str), freq="Q")
production_theta = pd.read_csv(COVID_THETA_PATH)

TARGET_COUNTRIES = ["Germany", "France", "Italy", "Spain"]
GPR_ORIGIN_COLS = {
    "Russia": "log1p_GPR_Russia",
    "China": "log1p_GPR_China",
    "MENA": "log1p_GPR_Middle_East",
    "Western": "log1p_GPR_Western_Bloc",
}
GPR_ORIGINS_COLS = list(GPR_ORIGIN_COLS.values())
origins = list(GPR_ORIGIN_COLS.keys())
CHOLESKY_ORDER_CORE = ["dlog_Brent", "dlog_Gas", "dlog_HICP", "dlog_IPI", "CLIFS_level"]

LAG = 2
IRF_HORIZON = 20
CI_LOW_68, CI_HIGH_68 = 16, 84
HORIZONS_TO_CHECK = {"impact": 0, "1yr": 4, "2yr": 8}
REPS, BURN, SEED = 10000, 5000, 42

theta_lookup = {
    (row["country"], row["gpr_origin"]):
    (row["s0_2020Q1"], row["s1_2020Q2"], row["s2_2020Q3"], row["rho"])
    for _, row in production_theta.iterrows()
}

with open(BASELINE_RESULTS_PATH, "rb") as f:
    bvar_results = pickle.load(f)

if len(bvar_results) != 16 or len(theta_lookup) != 16:
    raise RuntimeError("Expected 16 baseline systems and 16 frozen COVID parameter sets.")

print(f"Loaded {len(bvar_results)} baseline BVAR systems from notebook 05.")
print(f"Loaded {len(theta_lookup)} frozen COVID parameter sets from notebook 04.")

def build_truncated_covid_scale_path(quarters, s0, s1, s2, rho):
    """Identical to the function used in NB04/NB05/NB04 -- must not diverge."""
    periods = pd.PeriodIndex(pd.Series(quarters).astype(str), freq="Q")
    path = np.ones(len(periods), dtype=float)
    q0, q1, q2 = pd.Period("2020Q1", "Q"), pd.Period("2020Q2", "Q"), pd.Period("2020Q3", "Q")
    trunc = pd.Period("2021Q4", "Q")
    for idx, q in enumerate(periods):
        if q == q0:
            path[idx] = s0
        elif q == q1:
            path[idx] = s1
        elif q == q2:
            path[idx] = s2
        elif q > q2 and q < trunc:
            j = q.ordinal - q2.ordinal
            path[idx] = 1.0 + (s2 - 1.0) * (rho ** j)
    return path

def estimate_var_ols(Y, p):
    """OLS VAR(p). X = [const, lag1(all vars), lag2(all vars), ...]. Homoskedastic, no COVID scaling."""
    T, n = Y.shape
    X = np.column_stack([np.ones(T - p)] + [Y[p - i - 1: T - i - 1] for i in range(p)])
    Ydep = Y[p:]
    Beta = np.linalg.lstsq(X, Ydep, rcond=None)[0]
    Uhat = Ydep - X @ Beta
    Sigma = (Uhat.T @ Uhat) / (T - p)
    return Beta, Uhat, Sigma, X, Ydep

def estimate_ar1(x):
    """Univariate AR(1) coefficient via OLS. Used for the GPR prior mean."""
    y = x[1:]; x_lag = x[:-1]
    X = np.column_stack([np.ones(len(x_lag)), x_lag])
    return np.linalg.lstsq(X, y, rcond=None)[0][1]

def build_minnesota_prior_mixed(s, n, p, gpr_idx, phi_gpr,
                                 lam1=0.2, lam2=0.5, lam3=1.0, lam4=100.0):
    """Standard mixed prior mean: GPR own-lag1 = phi_gpr (AR(1) estimated), all
    other own-lag1 means = 0 (white noise, appropriate for the already-stationary
    log-differenced/bounded variables in the baseline specification)."""
    k = 1 + n * p
    B0 = np.zeros(n * k)
    H_diag = np.zeros(n * k)
    for i in range(n):
        base = i * k
        H_diag[base] = (s[i] * lam4) ** 2
        for l in range(1, p + 1):
            for j in range(n):
                col = base + 1 + (l - 1) * n + j
                if j == i:
                    H_diag[col] = (lam1 / (l ** lam3)) ** 2
                    if l == 1 and i == gpr_idx:
                        B0[col] = phi_gpr
                else:
                    H_diag[col] = ((s[i] * lam1 * lam2) / (s[j] * (l ** lam3))) ** 2
    return B0, np.diag(H_diag)

def IWPQ(v, ixpx, rng):
    """Draw from Inverse-Wishart(v, ixpx^{-1}). Constructs a Wishart(v, ixpx) draw
    as the sum of v outer products of independent Gaussian vectors with covariance
    ixpx, generated via the Cholesky factor of ixpx (direct Gaussian construction,
    not a Bartlett decomposition), then inverts to obtain the Inverse-Wishart draw.
    Uses ONLY the passed rng -- no calls to the NumPy global random state."""
    k = ixpx.shape[0]
    L = np.linalg.cholesky(ixpx)
    A = np.zeros((v, k))
    for i in range(v):
        A[i] = L @ rng.standard_normal(k)
    return np.linalg.inv(A.T @ A)


def bvar_minnesota_gibbs_final(Y, quarters, p, gpr_idx, theta,
                                S_matrix=None, own_lag1_means=None, exclude_quarters=None,
                                reps=REPS, burn=BURN, seed=SEED,
                                lam1=0.2, lam2=0.5, lam3=1.0, lam4=100.0):
    """
    Shared production sampler. n is inferred from Y.shape[1], so the SAME
    function serves the full 6-variable system, the 4-variable reduced
    system, reordered/relevelled variants, and crisis-exclusion diagnostics.

    Prior structure: beta ~ N(B0, H), independent of Sigma a priori.
    p(Sigma) ~ |Sigma|^{-(N+1)/2} when S_matrix=None (S=0), the improper
    Jeffreys-type prior used as the NB05 baseline. alpha is FIXED at 0.
    The Sigma-prior robustness in Block C changes S_matrix only.

    theta = (s0, s1, s2, rho): frozen COVID volatility parameters estimated
    once in NB04. They are never re-estimated here.

    own_lag1_means: optional array of length n for alternative own-lag-1
    prior means. GPR's own-lag-1 mean is always enforced to its estimated
    AR(1) coefficient.

    exclude_quarters: optional iterable of quarters to exclude from the
    likelihood. The original quarterly calendar is preserved. A VAR equation
    is excluded whenever its dependent quarter or any of its p lagged
    quarters belongs to the excluded set, preventing artificial lag jumps.

    Sampler: reps total Gibbs iterations, burn discarded, no thinning.
    """
    rng = np.random.default_rng(seed)
    T, n = Y.shape
    q = np.asarray(quarters).astype(str)

    # Complete VAR design: never physically delete quarters before constructing lags.
    X_raw = np.column_stack([np.ones(T - p)] + [Y[p - i - 1:T - i - 1] for i in range(p)])
    Ydep_raw = Y[p:]

    # Estimation-row mask.
    valid = np.ones(T - p, dtype=bool)

    if exclude_quarters is not None:
        excluded = set(str(x) for x in exclude_quarters)

        # Remove an equation if either y_t or any of its p lags lies in the excluded window.
        for lag in range(p + 1):
            valid &= ~np.isin(q[p - lag:T - lag], list(excluded))

        if valid.sum() <= 0:
            raise ValueError("No valid VAR observations remain after applying exclude_quarters.")

        # Prior scales from the unadjusted homoskedastic VAR on the same valid equations.
        X_prior = X_raw[valid]
        Ydep_prior = Ydep_raw[valid]
        Beta_prior = np.linalg.lstsq(X_prior, Ydep_prior, rcond=None)[0]
        U_prior = Ydep_prior - X_prior @ Beta_prior
        Sigma_baseline = U_prior.T @ U_prior / valid.sum()

        # AR(1) GPR prior mean without creating adjacency across the excluded interval.
        ar_valid = (~np.isin(q[1:], list(excluded))) & (~np.isin(q[:-1], list(excluded)))
        X_ar = np.column_stack([np.ones(ar_valid.sum()), Y[:-1, gpr_idx][ar_valid]])
        y_ar = Y[1:, gpr_idx][ar_valid]
        phi_gpr = np.linalg.lstsq(X_ar, y_ar, rcond=None)[0][1]

    else:
        # Exact baseline machinery when no observations are excluded.
        _, _, Sigma_baseline, _, _ = estimate_var_ols(Y, p)
        phi_gpr = estimate_ar1(Y[:, gpr_idx])

    s_prior = np.sqrt(np.diag(Sigma_baseline))

    # Frozen COVID volatility correction.
    s0, s1, s2, rho = theta
    full_scale = build_truncated_covid_scale_path(quarters, s0, s1, s2, rho)
    s_eff = full_scale[p:]

    X = X_raw[valid] / s_eff[valid, None]
    Ydep = Ydep_raw[valid] / s_eff[valid, None]
    T_eff = Ydep.shape[0]

    Beta_ols = np.linalg.lstsq(X, Ydep, rcond=None)[0]

    if own_lag1_means is None:
        B0, H = build_minnesota_prior_mixed(s_prior, n, p, gpr_idx, phi_gpr, lam1, lam2, lam3, lam4)
    else:
        means = np.asarray(own_lag1_means, dtype=float).copy()
        if len(means) != n:
            raise ValueError("own_lag1_means must have length n.")
        means[gpr_idx] = phi_gpr
        B0, H = build_minnesota_prior_mixed_general(s_prior, n, p, gpr_idx, means,
                                                     lam1, lam2, lam3, lam4)

    Hinv = np.linalg.inv(H)

    alpha = 0
    S = np.zeros((n, n)) if S_matrix is None else S_matrix

    betaols_vec = Beta_ols.T.reshape(-1)
    Sigma = (Ydep - X @ Beta_ols).T @ (Ydep - X @ Beta_ols) / T_eff

    k = 1 + n * p
    XtX = X.T @ X

    beta_draws = np.zeros((reps - burn, n, k))
    sigma_draws = np.zeros((reps - burn, n, n))

    for it in range(reps):
        SigInv = np.linalg.inv(Sigma)
        V = np.linalg.inv(Hinv + np.kron(SigInv, XtX))
        M = V @ (Hinv @ B0 + np.kron(SigInv, XtX) @ betaols_vec)

        beta_vec = M + np.linalg.cholesky(V) @ rng.standard_normal(n * k)
        Beta = beta_vec.reshape(n, k).T

        Uhat_it = Ydep - X @ Beta
        scale = Uhat_it.T @ Uhat_it + S
        Sigma = IWPQ(T_eff + alpha, np.linalg.inv(scale), rng)

        if it >= burn:
            beta_draws[it - burn] = Beta.T
            sigma_draws[it - burn] = Sigma

    return beta_draws, sigma_draws, phi_gpr


def build_minnesota_prior_mixed_general(s, n, p, gpr_idx, own_lag1_means,
                                         lam1=0.2, lam2=0.5, lam3=1.0, lam4=100.0):
    """Same as build_minnesota_prior_mixed, but the own-lag1 prior mean of
    every variable is supplied explicitly via own_lag1_means (length n),
    instead of hardcoding 0 for all non-GPR variables. Used by Block D."""
    k = 1 + n * p
    B0 = np.zeros(n * k)
    H_diag = np.zeros(n * k)
    for i in range(n):
        base = i * k
        H_diag[base] = (s[i] * lam4) ** 2
        for l in range(1, p + 1):
            for j in range(n):
                col = base + 1 + (l - 1) * n + j
                if j == i:
                    H_diag[col] = (lam1 / (l ** lam3)) ** 2
                    if l == 1:
                        B0[col] = own_lag1_means[i]
                else:
                    H_diag[col] = ((s[i] * lam1 * lam2) / (s[j] * (l ** lam3))) ** 2
    return B0, np.diag(H_diag)


def compute_structural_irfs(Phis, Sigma, horizon, n, p):
    Psi = [np.eye(n)]
    for j in range(1, horizon + 1):
        psi_j = np.zeros((n, n))
        for i in range(1, min(j, p) + 1):
            psi_j += Phis[i - 1] @ Psi[j - i]
        Psi.append(psi_j)
    Q = np.linalg.cholesky(Sigma)
    return np.array([psi @ Q for psi in Psi])

def phis_from_beta(Beta_T, n, p):
    return [Beta_T[:, 1 + i*n : 1 + (i+1)*n] for i in range(p)]

def bvar_structural_irfs(beta_draws, sigma_draws, n, p, horizon):
    n_draws = beta_draws.shape[0]
    Theta_draws = np.zeros((n_draws, horizon + 1, n, n))
    for d in range(n_draws):
        Phis = phis_from_beta(beta_draws[d], n, p)
        Theta_draws[d] = compute_structural_irfs(Phis, sigma_draws[d], horizon, n, p)
    return Theta_draws

def system_quarters(country, var_order):
    """Returns the aligned quarter labels for a (country, var_order) system,
    needed to build each system's COVID scale path."""
    sub = panel[panel["country"] == country].sort_values("quarter").reset_index(drop=True)
    return sub[["quarter"] + var_order].dropna()["quarter"].astype(str).reset_index(drop=True)



Mounted at /content/drive
Loaded 16 baseline BVAR systems from notebook 05.
Loaded 16 frozen COVID parameter sets from notebook 04.


## B. Reduced system

In [2]:
# ============================================================
# A1. RUN BVAR -- REDUCED SYSTEM (drops Brent, Gas): [GPR, HICP, IPI, CLIFS]
# ============================================================
# Uses the SAME frozen theta as the corresponding full-system estimate
# (no re-estimation of theta for the reduced system -- a deliberate
# simplification, documented as such).

CHOLESKY_ORDER_REDUCED = ["dlog_HICP", "dlog_IPI", "CLIFS_level"]

bvar_results_reduced = {}   # keyed by (country, origin) -> (Theta_draws, phi_gpr, var_order)

for country in TARGET_COUNTRIES:
    sub = panel[panel["country"] == country].sort_values("quarter").reset_index(drop=True)

    for gpr_col in GPR_ORIGINS_COLS:
        origin = next(o for o, col in GPR_ORIGIN_COLS.items() if col == gpr_col)
        var_order = [gpr_col] + CHOLESKY_ORDER_REDUCED  # n=4
        data_full = sub[["quarter"] + var_order].dropna()
        quarters = data_full["quarter"].astype(str).reset_index(drop=True)
        data = data_full[var_order].values
        theta = theta_lookup[(country, origin)]

        beta_draws, sigma_draws, phi_gpr = bvar_minnesota_gibbs_final(
            data, quarters, p=LAG, gpr_idx=0, theta=theta
        )
        Theta_draws = bvar_structural_irfs(beta_draws, sigma_draws, n=4, p=LAG, horizon=IRF_HORIZON)

        bvar_results_reduced[(country, origin)] = (Theta_draws, phi_gpr, var_order)

    print(f"{country}: reduced system done for all 4 origins.")

print(f"\nReduced system (4 vars) estimated for {len(bvar_results_reduced)} systems.")
print(f"Parameters per equation: {1 + 4*LAG} (vs. {1 + 6*LAG} in the full system)")
print(f"Obs-to-parameter ratio: ~{113/(1+4*LAG):.1f} (vs. ~{113/(1+6*LAG):.1f} in the full system)")

Germany: reduced system done for all 4 origins.
France: reduced system done for all 4 origins.
Italy: reduced system done for all 4 origins.
Spain: reduced system done for all 4 origins.

Reduced system (4 vars) estimated for 16 systems.
Parameters per equation: 9 (vs. 13 in the full system)
Obs-to-parameter ratio: ~12.6 (vs. ~8.7 in the full system)


In [3]:
# ============================================================
# A2. BVAR (6-VAR BASELINE) vs. BVAR (4-VAR REDUCED) -- DIRECT COMPARISON
# ============================================================
# Redesigned: no longer depends on D3_classical_reduced.pkl (not
# produced by the current NB04 pipeline). Compares the two BVAR
# specifications DIRECTLY for the three shared variables (HICP, IPI,
# CLIFS), both already COVID-corrected and alpha=0.
#
# Question: does reducing system dimensionality increase posterior
# precision or materially alter inference on the macro-financial
# responses -- not "does it explain a classical-vs-BVAR retention gap".

reduced_comparison_rows = []

for (country, origin), (Theta_red, phi_gpr_red, var_order_red) in bvar_results_reduced.items():
    Theta_full, phi_gpr_full, var_order_full = bvar_results[(country, origin)]

    med_red = np.median(Theta_red, axis=0)
    low68_red = np.percentile(Theta_red, CI_LOW_68, axis=0)
    high68_red = np.percentile(Theta_red, CI_HIGH_68, axis=0)

    med_full = np.median(Theta_full, axis=0)
    low68_full = np.percentile(Theta_full, CI_LOW_68, axis=0)
    high68_full = np.percentile(Theta_full, CI_HIGH_68, axis=0)

    for var_name in ["dlog_HICP", "dlog_IPI", "CLIFS_level"]:
        i_red = var_order_red.index(var_name)
        i_full = var_order_full.index(var_name)

        for h_label, h in HORIZONS_TO_CHECK.items():
            sig_red = (low68_red[h, i_red, 0] > 0) or (high68_red[h, i_red, 0] < 0)
            sig_full = (low68_full[h, i_full, 0] > 0) or (high68_full[h, i_full, 0] < 0)
            width_red = high68_red[h, i_red, 0] - low68_red[h, i_red, 0]
            width_full = high68_full[h, i_full, 0] - low68_full[h, i_full, 0]

            reduced_comparison_rows.append({
                "country": country, "gpr_origin": origin,
                "response_var": var_name, "horizon": h_label,
                "median_6var": round(med_full[h, i_full, 0], 5),
                "median_4var": round(med_red[h, i_red, 0], 5),
                "same_sign": np.sign(med_full[h, i_full, 0]) == np.sign(med_red[h, i_red, 0]),
                "sig_68_6var": sig_full, "sig_68_4var": sig_red,
                "significance_changed": sig_full != sig_red,
                "width_ratio_4var_over_6var": round(width_red / width_full, 3) if width_full > 0 else np.nan,
            })

reduced_df = pd.DataFrame(reduced_comparison_rows)

print(f"Total comparisons: {len(reduced_df)} (16 systems x 3 shared variables x 3 horizons)\n")
print(f"Sign agreement: {100*reduced_df['same_sign'].mean():.1f}%")
print(f"Significance changes: {reduced_df['significance_changed'].sum()} / {len(reduced_df)}")
print(f"Mean 68% band width ratio (4-var / 6-var): " f"{reduced_df['width_ratio_4var_over_6var'].mean():.3f}")
print(reduced_df["width_ratio_4var_over_6var"].quantile([0.25, 0.50, 0.75]))
print("(<1 means the reduced system is MORE precise; >1 means LESS precise.)\n")

print("Cases where significance status changed between the two specifications:")
display(reduced_df[reduced_df["significance_changed"]][
    ["country", "gpr_origin", "response_var", "horizon", "median_6var", "median_4var", "sig_68_6var", "sig_68_4var"]
])

Total comparisons: 144 (16 systems x 3 shared variables x 3 horizons)

Sign agreement: 94.4%
Significance changes: 5 / 144
Mean 68% band width ratio (4-var / 6-var): 0.989
0.25    0.961
0.50    0.985
0.75    1.010
Name: width_ratio_4var_over_6var, dtype: float64
(<1 means the reduced system is MORE precise; >1 means LESS precise.)

Cases where significance status changed between the two specifications:


,country,gpr_origin,response_var,horizon,median_6var,median_4var,sig_68_6var,sig_68_4var
17,Germany,China,CLIFS_level,2yr,-0.00409,-0.00322,True,False
25,Germany,MENA,CLIFS_level,1yr,-0.00480,-0.00467,False,True
27,Germany,Western,dlog_HICP,impact,0.00041,0.00052,False,True
69,France,Western,CLIFS_level,impact,0.00523,0.00491,True,False
114,Spain,Russia,CLIFS_level,impact,-0.00611,-0.00500,True,False


## C. Structural identification and Gas-response robustness

In [4]:
# ============================================================
# B1. BRENT/GAS ORDERING ROBUSTNESS -- ALL 16 SYSTEMS
# Inverts Brent/Gas relative order only (GPR stays first).
# Tests the mechanical hypothesis for Gas's systematic negative sign
# at impact.
# ============================================================
var_order_altB_core = ["dlog_Gas", "dlog_Brent", "dlog_HICP", "dlog_IPI", "CLIFS_level"]

gas_sign_comparison = []

for country in TARGET_COUNTRIES:
    sub = panel[panel["country"] == country].sort_values("quarter").reset_index(drop=True)
    for gpr_col in GPR_ORIGINS_COLS:
        origin = next(o for o, col in GPR_ORIGIN_COLS.items() if col == gpr_col)
        var_order = [gpr_col] + var_order_altB_core
        data_full = sub[["quarter"] + var_order].dropna()
        quarters = data_full["quarter"].astype(str).reset_index(drop=True)
        data = data_full[var_order].values
        theta = theta_lookup[(country, origin)]

        beta_draws, sigma_draws, phi_gpr = bvar_minnesota_gibbs_final(
            data, quarters, p=LAG, gpr_idx=0, theta=theta
        )
        Theta_draws = bvar_structural_irfs(beta_draws, sigma_draws, n=6, p=LAG, horizon=IRF_HORIZON)
        median_altB = np.median(Theta_draws, axis=0)

        Theta_baseline, _, var_order_base = bvar_results[(country, origin)]
        gas_idx_base = var_order_base.index("dlog_Gas")
        median_base = np.median(Theta_baseline, axis=0)

        gas_idx_altB = var_order.index("dlog_Gas")

        gas_sign_comparison.append({
            "country": country, "gpr_origin": origin,
            "gas_baseline_impact": round(median_base[0, gas_idx_base, 0], 5),
            "gas_gas_first_impact": round(median_altB[0, gas_idx_altB, 0], 5),
            "sign_flip": np.sign(median_base[0, gas_idx_base, 0]) != np.sign(median_altB[0, gas_idx_altB, 0]),
        })
    print(f"{country}: done (all 4 origins).")

gas_sign_df = pd.DataFrame(gas_sign_comparison)
display(gas_sign_df)
print(f"\nSign flips when Gas is ordered before Brent: {gas_sign_df['sign_flip'].sum()} of 16")

Germany: done (all 4 origins).
France: done (all 4 origins).
Italy: done (all 4 origins).
Spain: done (all 4 origins).


,country,gpr_origin,gas_baseline_impact,gas_gas_first_impact,sign_flip
0,Germany,Russia,-0.02308,-0.02310,False
1,Germany,China,-0.01052,-0.01058,False
2,Germany,MENA,-0.00975,-0.01033,False
3,Germany,Western,-0.03240,-0.03276,False
4,France,Russia,-0.02596,-0.02597,False
5,France,China,-0.00976,-0.01028,False
6,France,MENA,-0.00006,-0.00042,False
7,France,Western,-0.02577,-0.02625,False
8,Italy,Russia,-0.03100,-0.03135,False
9,Italy,China,-0.01508,-0.01553,False



Sign flips when Gas is ordered before Brent: 0 of 16


In [5]:
# ============================================================
# B2. GPR-FIRST IDENTIFICATION ROBUSTNESS -- ALL 16 SYSTEMS
# Baseline: [GPR, Brent, Gas, HICP, IPI, CLIFS]
# Alternative: [Brent, Gas, GPR, HICP, IPI, CLIFS]
# Tests sensitivity to GPR-first while preserving Brent-before-Gas.
# ============================================================

gpr_energy_ordering_table = []
bvar_results_energy_first = {}

for country in TARGET_COUNTRIES:
    sub = panel[panel["country"] == country].sort_values("quarter").reset_index(drop=True)

    for gpr_col in GPR_ORIGINS_COLS:
        origin = next(o for o, col in GPR_ORIGIN_COLS.items() if col == gpr_col)
        var_order_alt = ["dlog_Brent", "dlog_Gas", gpr_col, "dlog_HICP", "dlog_IPI", "CLIFS_level"]

        data_full = sub[["quarter"] + var_order_alt].dropna()
        quarters = data_full["quarter"].astype(str).reset_index(drop=True)
        data = data_full[var_order_alt].values

        theta = theta_lookup[(country, origin)]
        gpr_idx_alt = var_order_alt.index(gpr_col)

        beta_draws, sigma_draws, phi_gpr = bvar_minnesota_gibbs_final(
            data, quarters, p=LAG, gpr_idx=gpr_idx_alt, theta=theta
        )

        Theta_alt = bvar_structural_irfs(beta_draws, sigma_draws, n=6, p=LAG, horizon=IRF_HORIZON)
        bvar_results_energy_first[(country, origin)] = (Theta_alt, phi_gpr, var_order_alt)

        Theta_base, _, var_order_base = bvar_results[(country, origin)]

        gas_idx_base = var_order_base.index("dlog_Gas")
        gpr_idx_base = var_order_base.index(gpr_col)
        gas_idx_alt = var_order_alt.index("dlog_Gas")

        median_base = np.median(Theta_base, axis=0)
        median_alt = np.median(Theta_alt, axis=0)

        row_base = {"country": country, "gpr_origin": origin, "ordering": "GPR_first"}
        row_alt = {"country": country, "gpr_origin": origin, "ordering": "energy_first"}

        for h in range(9):
            row_base[f"h{h}"] = round(median_base[h, gas_idx_base, gpr_idx_base], 5)
            row_alt[f"h{h}"] = round(median_alt[h, gas_idx_alt, gpr_idx_alt], 5)

        gpr_energy_ordering_table += [row_base, row_alt]

    print(f"{country}: done (all 4 origins).")

gpr_energy_ordering_df = pd.DataFrame(gpr_energy_ordering_table)

pd.set_option("display.max_columns", None)
display(gpr_energy_ordering_df)

Germany: done (all 4 origins).
France: done (all 4 origins).
Italy: done (all 4 origins).
Spain: done (all 4 origins).


,country,gpr_origin,ordering,h0,h1,h2,h3,h4,h5,h6,h7,h8
0,Germany,Russia,GPR_first,-0.02308,-0.00571,-0.00439,-0.00802,-0.00954,-0.00882,-0.00783,-0.00707,-0.00648
1,Germany,Russia,energy_first,0.00000,-0.00459,-0.00877,-0.00991,-0.00952,-0.00866,-0.00780,-0.00703,-0.00639
2,Germany,China,GPR_first,-0.01052,0.00384,0.00506,-0.00292,-0.00641,-0.00593,-0.00495,-0.00428,-0.00376
3,Germany,China,energy_first,0.00000,-0.00437,-0.00413,-0.00548,-0.00607,-0.00558,-0.00487,-0.00422,-0.00369
4,Germany,MENA,GPR_first,-0.00975,-0.01425,-0.01112,-0.00582,-0.00294,-0.00208,-0.00174,-0.00139,-0.00107
5,Germany,MENA,energy_first,0.00000,-0.00532,-0.00619,-0.00510,-0.00366,-0.00271,-0.00207,-0.00165,-0.00128
6,Germany,Western,GPR_first,-0.03240,-0.01634,-0.00825,-0.00630,-0.00541,-0.00483,-0.00437,-0.00396,-0.00356
7,Germany,Western,energy_first,0.00000,-0.00727,-0.00854,-0.00758,-0.00612,-0.00538,-0.00477,-0.00427,-0.00381
8,France,Russia,GPR_first,-0.02596,0.00006,0.00111,-0.00422,-0.00626,-0.00576,-0.00495,-0.00435,-0.00381
9,France,Russia,energy_first,0.00000,-0.00115,-0.00587,-0.00736,-0.00686,-0.00583,-0.00489,-0.00423,-0.00360


In [6]:
# ============================================================
# B2. SUMMARY OF GAS RESPONSE UNDER ENERGY-FIRST ORDERING
# Impact is excluded because it is mechanically zero by Cholesky.
# ============================================================

for h in [1, 2, 4, 8]:
    vals = gpr_energy_ordering_df[gpr_energy_ordering_df["ordering"] == "energy_first"][f"h{h}"]
    print(f"h{h}: negative Gas response in {(vals < 0).sum()} / 16 systems")

h1: negative Gas response in 16 / 16 systems
h2: negative Gas response in 16 / 16 systems
h4: negative Gas response in 15 / 16 systems
h8: negative Gas response in 11 / 16 systems


In [7]:
# ============================================================
# B4. IS THE NEGATIVE GAS SIGN DRIVEN BY THE 2021-2022 CRISIS?
# Excludes 2021Q4-2022Q4 while preserving the quarterly lag structure.
# Frozen theta, S=0, alpha=0 and all other settings match baseline.
# Diagnostic only: tests whether the negative Gas impact response persists.
# ============================================================

CRISIS_START, CRISIS_END = "2021Q4", "2022Q4"
CRISIS_QUARTERS = pd.period_range(CRISIS_START, CRISIS_END, freq="Q").astype(str).tolist()

gas_no_crisis_comparison = []
bvar_results_no_crisis = {}

for country in TARGET_COUNTRIES:
    sub = panel[panel["country"] == country].sort_values("quarter").reset_index(drop=True)

    for gpr_col in GPR_ORIGINS_COLS:
        origin = next(o for o, col in GPR_ORIGIN_COLS.items() if col == gpr_col)
        var_order = [gpr_col] + CHOLESKY_ORDER_CORE

        data_full = sub[["quarter"] + var_order].dropna()
        quarters = data_full["quarter"].astype(str).reset_index(drop=True)
        data = data_full[var_order].values

        theta = theta_lookup[(country, origin)]

        beta_draws, sigma_draws, phi_gpr = bvar_minnesota_gibbs_final(
            data, quarters, p=LAG, gpr_idx=0, theta=theta, exclude_quarters=CRISIS_QUARTERS
        )

        Theta_excl = bvar_structural_irfs(beta_draws, sigma_draws, n=6, p=LAG, horizon=IRF_HORIZON)
        bvar_results_no_crisis[(country, origin)] = (Theta_excl, phi_gpr, var_order)

        Theta_base, _, var_order_base = bvar_results[(country, origin)]

        gas_idx_base = var_order_base.index("dlog_Gas")
        gas_idx_excl = var_order.index("dlog_Gas")

        median_base = np.median(Theta_base, axis=0)
        median_excl = np.median(Theta_excl, axis=0)

        gas_no_crisis_comparison.append({
            "country": country,
            "gpr_origin": origin,
            "gas_full_sample": median_base[0, gas_idx_base, 0],
            "gas_excl_crisis": median_excl[0, gas_idx_excl, 0],
        })

    print(f"{country}: done (all 4 origins).")

gas_crisis_df = pd.DataFrame(gas_no_crisis_comparison)
gas_crisis_df["still_negative"] = gas_crisis_df["gas_excl_crisis"] < 0

display(gas_crisis_df.round(5))

print(f"\nStill negative excluding {CRISIS_START}-{CRISIS_END}: "
      f"{gas_crisis_df['still_negative'].sum()} of 16")

Germany: done (all 4 origins).
France: done (all 4 origins).
Italy: done (all 4 origins).
Spain: done (all 4 origins).


,country,gpr_origin,gas_full_sample,gas_excl_crisis,still_negative
0,Germany,Russia,-0.02308,-0.01602,True
1,Germany,China,-0.01052,-0.00514,True
2,Germany,MENA,-0.00975,-0.00166,True
3,Germany,Western,-0.03240,-0.01929,True
4,France,Russia,-0.02596,-0.01880,True
5,France,China,-0.00976,-0.00769,True
6,France,MENA,-0.00006,0.00287,False
7,France,Western,-0.02577,-0.01819,True
8,Italy,Russia,-0.03100,-0.02015,True
9,Italy,China,-0.01508,-0.00984,True



Still negative excluding 2021Q4-2022Q4: 15 of 16


### B4. Diagnostic interpretation: excluding the 2021--2022 European gas crisis

The baseline BVAR was re-estimated after excluding 2021Q4--2022Q4 while preserving the quarterly lag structure. Because the model contains two lags, equations for 2023Q1 and 2023Q2 are also excluded from estimation since their regressors contain observations from the excluded crisis window.

The median impact response of Gas to a GPR shock remains negative in 15 of the 16 country-origin systems. The only sign reversal occurs for France-MENA, where the baseline response was already effectively zero (-0.00006).

For most systems, excluding the crisis reduces the absolute magnitude of the negative Gas response. For example, Germany-Russia changes from -0.02308 to -0.01602, Italy-Russia from -0.03100 to -0.02015, and Spain-Western from -0.03823 to -0.02444.

The exercise therefore suggests that the extraordinary European gas-crisis episode amplified the estimated negative Gas response but does not by itself account for its presence. This is a diagnostic robustness exercise and should not be interpreted as identifying the causal effect of the 2021--2022 crisis on the estimated GPR-Gas relationship.

## D. Covariance-prior sensitivity

In [8]:
# ============================================================
# C1. SIGMA PRIOR ROBUSTNESS -- S=I vs. S=0, BOTH alpha=0
# Baseline: S=0, alpha=0 (NB05 production specification).
# Robustness: S=I, alpha=0. ONLY S changes.
# ============================================================

# ---- Baseline S=0, alpha=0: recompute significance counts from bvar_results ----

synthesis_rows_S0 = []

for (country, origin), (Theta_draws, phi_gpr, var_order) in bvar_results.items():
    Theta_low68 = np.percentile(Theta_draws, CI_LOW_68, axis=0)
    Theta_high68 = np.percentile(Theta_draws, CI_HIGH_68, axis=0)

    for i, var_name in enumerate(var_order[1:], start=1):
        for h_label, h in HORIZONS_TO_CHECK.items():
            sig = (Theta_low68[h, i, 0] > 0) or (Theta_high68[h, i, 0] < 0)
            synthesis_rows_S0.append({
                "country": country, "gpr_origin": origin,
                "response_var": var_name, "horizon": h_label, "significant_68": sig
            })

synthesis_df_S0 = pd.DataFrame(synthesis_rows_S0)

block_data_S0 = {}
for country in TARGET_COUNTRIES:
    block_data_S0[country] = {}
    for origin in origins:
        mask = (
            (synthesis_df_S0["country"] == country) &
            (synthesis_df_S0["gpr_origin"] == origin) &
            (synthesis_df_S0["horizon"].isin(["impact", "1yr"]))
        )
        block_data_S0[country][origin] = synthesis_df_S0.loc[mask, "significant_68"].values

observed_counts_S0 = {
    o: sum(block_data_S0[c][o].sum() for c in TARGET_COUNTRIES)
    for o in origins
}

# ---- Robustness S=I, alpha=0: re-run all 16 systems ----

bvar_results_SI = {}

for country in TARGET_COUNTRIES:
    sub = panel[panel["country"] == country].sort_values("quarter").reset_index(drop=True)

    for gpr_col in GPR_ORIGINS_COLS:
        origin = next(o for o, col in GPR_ORIGIN_COLS.items() if col == gpr_col)
        var_order = [gpr_col] + CHOLESKY_ORDER_CORE

        data_full = sub[["quarter"] + var_order].dropna()
        quarters = data_full["quarter"].astype(str).reset_index(drop=True)
        data = data_full[var_order].values

        theta = theta_lookup[(country, origin)]

        beta_draws, sigma_draws, phi_gpr = bvar_minnesota_gibbs_final(
            data, quarters, p=LAG, gpr_idx=0, theta=theta, S_matrix=np.eye(len(var_order))
        )

        Theta_draws = bvar_structural_irfs(beta_draws, sigma_draws, n=6, p=LAG, horizon=IRF_HORIZON)
        bvar_results_SI[(country, origin)] = (Theta_draws, phi_gpr, var_order)

    print(f"{country}: S=I done (all 4 origins).")

# ---- S=I significance counts ----

synthesis_rows_SI = []

for (country, origin), (Theta_draws, phi_gpr, var_order) in bvar_results_SI.items():
    Theta_low68 = np.percentile(Theta_draws, CI_LOW_68, axis=0)
    Theta_high68 = np.percentile(Theta_draws, CI_HIGH_68, axis=0)

    for i, var_name in enumerate(var_order[1:], start=1):
        for h_label, h in HORIZONS_TO_CHECK.items():
            sig = (Theta_low68[h, i, 0] > 0) or (Theta_high68[h, i, 0] < 0)
            synthesis_rows_SI.append({
                "country": country, "gpr_origin": origin,
                "response_var": var_name, "horizon": h_label, "significant_68": sig
            })

synthesis_df_SI = pd.DataFrame(synthesis_rows_SI)

block_data_SI = {}
for country in TARGET_COUNTRIES:
    block_data_SI[country] = {}
    for origin in origins:
        mask = (
            (synthesis_df_SI["country"] == country) &
            (synthesis_df_SI["gpr_origin"] == origin) &
            (synthesis_df_SI["horizon"].isin(["impact", "1yr"]))
        )
        block_data_SI[country][origin] = synthesis_df_SI.loc[mask, "significant_68"].values

observed_counts_SI = {
    o: sum(block_data_SI[c][o].sum() for c in TARGET_COUNTRIES)
    for o in origins
}

print("\nS=0, alpha=0 observed counts:", observed_counts_S0)
print("S=I, alpha=0 observed counts:", observed_counts_SI)

Germany: S=I done (all 4 origins).
France: S=I done (all 4 origins).
Italy: S=I done (all 4 origins).
Spain: S=I done (all 4 origins).

S=0, alpha=0 observed counts: {'Russia': np.int64(23), 'China': np.int64(15), 'MENA': np.int64(11), 'Western': np.int64(11)}
S=I, alpha=0 observed counts: {'Russia': np.int64(7), 'China': np.int64(4), 'MENA': np.int64(3), 'Western': np.int64(4)}


In [9]:
# ============================================================
# C2. INFERENCE ROBUSTNESS TO THE SIGMA PRIOR
# Compare S=0 vs S=I (alpha=0 in both):
# signs, 68% significance, posterior band widths, and
# exact origin-heterogeneity permutation results.
# ============================================================

sigma_prior_rows = []

for key in bvar_results:
    country, origin = key

    Theta_S0, _, var_order_S0 = bvar_results[key]
    Theta_SI, _, var_order_SI = bvar_results_SI[key]

    if var_order_S0 != var_order_SI:
        raise ValueError(f"Variable ordering differs for {country}-{origin}.")

    med_S0 = np.median(Theta_S0, axis=0)
    med_SI = np.median(Theta_SI, axis=0)

    low_S0 = np.percentile(Theta_S0, CI_LOW_68, axis=0)
    high_S0 = np.percentile(Theta_S0, CI_HIGH_68, axis=0)

    low_SI = np.percentile(Theta_SI, CI_LOW_68, axis=0)
    high_SI = np.percentile(Theta_SI, CI_HIGH_68, axis=0)

    for i, var_name in enumerate(var_order_S0[1:], start=1):
        for h_label, h in HORIZONS_TO_CHECK.items():

            median0 = med_S0[h, i, 0]
            medianI = med_SI[h, i, 0]

            sig0 = (low_S0[h, i, 0] > 0) or (high_S0[h, i, 0] < 0)
            sigI = (low_SI[h, i, 0] > 0) or (high_SI[h, i, 0] < 0)

            width0 = high_S0[h, i, 0] - low_S0[h, i, 0]
            widthI = high_SI[h, i, 0] - low_SI[h, i, 0]

            sigma_prior_rows.append({
                "country": country,
                "gpr_origin": origin,
                "response_var": var_name,
                "horizon": h_label,
                "median_S0": median0,
                "median_SI": medianI,
                "same_sign": np.sign(median0) == np.sign(medianI),
                "sig_S0": sig0,
                "sig_SI": sigI,
                "sig_change": sig0 != sigI,
                "width_S0": width0,
                "width_SI": widthI,
                "width_ratio_SI_over_S0": widthI / width0 if width0 > 0 else np.nan,
            })

sigma_prior_df = pd.DataFrame(sigma_prior_rows)

# ---- Overall inference comparison ----

n_comp = len(sigma_prior_df)
sign_agreement = sigma_prior_df["same_sign"].mean()
n_sig_changes = sigma_prior_df["sig_change"].sum()

width_ratio = sigma_prior_df["width_ratio_SI_over_S0"]

print(f"Total comparisons: {n_comp}")
print(f"Sign agreement S=0 vs S=I: {100 * sign_agreement:.1f}%")
print(f"Significance changes: {n_sig_changes} of {n_comp}")

print("\n68% band-width ratio (S=I / S=0):")
print(f"Mean:   {width_ratio.mean():.3f}")
print(f"Q25:    {width_ratio.quantile(0.25):.3f}")
print(f"Median: {width_ratio.median():.3f}")
print(f"Q75:    {width_ratio.quantile(0.75):.3f}")

# ---- Show individual significance changes ----

changed_sigma_prior = sigma_prior_df.loc[
    sigma_prior_df["sig_change"],
    ["country", "gpr_origin", "response_var", "horizon",
     "median_S0", "median_SI", "sig_S0", "sig_SI",
     "width_ratio_SI_over_S0"]
].copy()

print("\nCases with a change in 68% significance:")
display(changed_sigma_prior.round(5))


# ============================================================
# Exact origin-heterogeneity test under S=I
# Same 4^4 = 256 enumeration used for the baseline.
# Impact + 1-year responses only.
# ============================================================

from itertools import product

null_counts_SI = []

for assignment in product(origins, repeat=len(TARGET_COUNTRIES)):
    total = sum(
        block_data_SI[country][origin].sum()
        for country, origin in zip(TARGET_COUNTRIES, assignment)
    )
    null_counts_SI.append(total)

null_counts_SI = np.asarray(null_counts_SI)

permutation_SI_rows = []

for origin in origins:
    observed = observed_counts_SI[origin]
    p_value = np.mean(null_counts_SI >= observed)

    permutation_SI_rows.append({
        "gpr_origin": origin,
        "count_S0": observed_counts_S0[origin],
        "count_SI": observed_counts_SI[origin],
        "p_value_SI": p_value,
    })

permutation_SI_df = pd.DataFrame(permutation_SI_rows)

print("\nOrigin heterogeneity under S=I, alpha=0:")
print(f"Exact assignments: {len(null_counts_SI)}")
display(permutation_SI_df.round(6))

Total comparisons: 240
Sign agreement S=0 vs S=I: 87.9%
Significance changes: 66 of 240

68% band-width ratio (S=I / S=0):
Mean:   3.284
Q25:    1.175
Median: 1.391
Q75:    2.099

Cases with a change in 68% significance:


,country,gpr_origin,response_var,horizon,median_S0,median_SI,sig_S0,sig_SI,width_ratio_SI_over_S0
1,Germany,Russia,dlog_Brent,1yr,-0.00847,-0.00660,True,False,1.65490
4,Germany,Russia,dlog_Gas,1yr,-0.00954,-0.00855,True,False,1.47206
5,Germany,Russia,dlog_Gas,2yr,-0.00648,-0.00561,True,False,1.27408
6,Germany,Russia,dlog_HICP,impact,0.00065,0.00095,True,False,23.23309
7,Germany,Russia,dlog_HICP,1yr,0.00047,0.00001,True,False,1.59574
...,...,...,...,...,...,...,...,...,...
224,Spain,MENA,CLIFS_level,2yr,-0.00189,-0.00030,True,False,0.70100
235,Spain,Western,dlog_IPI,1yr,0.00113,0.00022,True,False,2.06031
236,Spain,Western,dlog_IPI,2yr,0.00026,0.00003,True,False,0.96022
238,Spain,Western,CLIFS_level,1yr,-0.00688,-0.00304,True,False,1.23893



Origin heterogeneity under S=I, alpha=0:
Exact assignments: 256


,gpr_origin,count_S0,count_SI,p_value_SI
0,Russia,23,7,0.078125
1,China,15,4,0.718750
2,MENA,11,3,1.000000
3,Western,11,4,0.718750


In [10]:
# ============================================================
# C3. SCALE-AWARE SIGMA PRIOR ROBUSTNESS
# Baseline: S=0, alpha=0.
# Alternative: S=diag(Sigma_OLS), alpha=0.
#
# Sigma_OLS is estimated from the corresponding unadjusted
# homoskedastic VAR, so the scale matrix respects the empirical
# residual scale of each equation. ONLY S changes.
# ============================================================

bvar_results_Sscaled = {}

for country in TARGET_COUNTRIES:
    sub = panel[panel["country"] == country].sort_values("quarter").reset_index(drop=True)

    for gpr_col in GPR_ORIGINS_COLS:
        origin = next(o for o, col in GPR_ORIGIN_COLS.items() if col == gpr_col)
        var_order = [gpr_col] + CHOLESKY_ORDER_CORE

        data_full = sub[["quarter"] + var_order].dropna()
        quarters = data_full["quarter"].astype(str).reset_index(drop=True)
        data = data_full[var_order].values

        theta = theta_lookup[(country, origin)]

        # Scale-aware S from the same unadjusted homoskedastic VAR
        # used for the baseline Minnesota prior scales.
        _, _, Sigma_ols, _, _ = estimate_var_ols(data, LAG)
        S_scaled = np.diag(np.diag(Sigma_ols))

        beta_draws, sigma_draws, phi_gpr = bvar_minnesota_gibbs_final(
            data, quarters, p=LAG, gpr_idx=0, theta=theta, S_matrix=S_scaled
        )

        Theta_draws = bvar_structural_irfs(
            beta_draws, sigma_draws, n=6, p=LAG, horizon=IRF_HORIZON
        )

        bvar_results_Sscaled[(country, origin)] = (Theta_draws, phi_gpr, var_order)

    print(f"{country}: scaled-S done (all 4 origins).")

Germany: scaled-S done (all 4 origins).
France: scaled-S done (all 4 origins).
Italy: scaled-S done (all 4 origins).
Spain: scaled-S done (all 4 origins).


In [11]:
# ============================================================
# C4. SCALE-AWARE SIGMA PRIOR: INFERENCE COMPARISON
# S=0 vs S=diag(Sigma_OLS), alpha=0 in both.
#
# Compare:
# - posterior-median signs
# - 68% significance
# - credible-band widths
# - origin heterogeneity
# ============================================================

scaled_prior_rows = []

for key in bvar_results:
    country, origin = key

    Theta_S0, _, var_order_S0 = bvar_results[key]
    Theta_scaled, _, var_order_scaled = bvar_results_Sscaled[key]

    if var_order_S0 != var_order_scaled:
        raise ValueError(f"Variable ordering differs for {country}-{origin}.")

    med_S0 = np.median(Theta_S0, axis=0)
    med_scaled = np.median(Theta_scaled, axis=0)

    low_S0 = np.percentile(Theta_S0, CI_LOW_68, axis=0)
    high_S0 = np.percentile(Theta_S0, CI_HIGH_68, axis=0)

    low_scaled = np.percentile(Theta_scaled, CI_LOW_68, axis=0)
    high_scaled = np.percentile(Theta_scaled, CI_HIGH_68, axis=0)

    for i, var_name in enumerate(var_order_S0[1:], start=1):
        for h_label, h in HORIZONS_TO_CHECK.items():

            median0 = med_S0[h, i, 0]
            medianS = med_scaled[h, i, 0]

            sig0 = (low_S0[h, i, 0] > 0) or (high_S0[h, i, 0] < 0)
            sigS = (low_scaled[h, i, 0] > 0) or (high_scaled[h, i, 0] < 0)

            width0 = high_S0[h, i, 0] - low_S0[h, i, 0]
            widthS = high_scaled[h, i, 0] - low_scaled[h, i, 0]

            scaled_prior_rows.append({
                "country": country,
                "gpr_origin": origin,
                "response_var": var_name,
                "horizon": h_label,
                "median_S0": median0,
                "median_scaled": medianS,
                "same_sign": np.sign(median0) == np.sign(medianS),
                "sig_S0": sig0,
                "sig_scaled": sigS,
                "sig_change": sig0 != sigS,
                "width_S0": width0,
                "width_scaled": widthS,
                "width_ratio_scaled_over_S0": widthS / width0 if width0 > 0 else np.nan,
            })

scaled_prior_df = pd.DataFrame(scaled_prior_rows)

# ---- Overall inference comparison ----

n_comp = len(scaled_prior_df)
sign_agreement = scaled_prior_df["same_sign"].mean()
n_sig_changes = scaled_prior_df["sig_change"].sum()
width_ratio = scaled_prior_df["width_ratio_scaled_over_S0"]

print(f"Total comparisons: {n_comp}")
print(f"Sign agreement S=0 vs scaled-S: {100 * sign_agreement:.1f}%")
print(f"Significance changes: {n_sig_changes} of {n_comp}")

print("\n68% band-width ratio (scaled-S / S=0):")
print(f"Mean:   {width_ratio.mean():.3f}")
print(f"Q25:    {width_ratio.quantile(0.25):.3f}")
print(f"Median: {width_ratio.median():.3f}")
print(f"Q75:    {width_ratio.quantile(0.75):.3f}")

# ---- Individual significance changes ----

changed_scaled_prior = scaled_prior_df.loc[
    scaled_prior_df["sig_change"],
    ["country", "gpr_origin", "response_var", "horizon",
     "median_S0", "median_scaled", "sig_S0", "sig_scaled",
     "width_ratio_scaled_over_S0"]
].copy()

print("\nCases with a change in 68% significance:")
display(changed_scaled_prior.round(5))


# ============================================================
# Origin heterogeneity under scaled-S
# Exact 4^4 = 256 enumeration, impact + 1-year horizons.
# ============================================================

synthesis_rows_scaled = []

for (country, origin), (Theta_draws, _, var_order) in bvar_results_Sscaled.items():
    low68 = np.percentile(Theta_draws, CI_LOW_68, axis=0)
    high68 = np.percentile(Theta_draws, CI_HIGH_68, axis=0)

    for i, var_name in enumerate(var_order[1:], start=1):
        for h_label, h in HORIZONS_TO_CHECK.items():
            sig = (low68[h, i, 0] > 0) or (high68[h, i, 0] < 0)

            synthesis_rows_scaled.append({
                "country": country,
                "gpr_origin": origin,
                "response_var": var_name,
                "horizon": h_label,
                "significant_68": sig,
            })

synthesis_df_scaled = pd.DataFrame(synthesis_rows_scaled)

block_data_scaled = {}

for country in TARGET_COUNTRIES:
    block_data_scaled[country] = {}

    for origin in origins:
        mask = (
            (synthesis_df_scaled["country"] == country) &
            (synthesis_df_scaled["gpr_origin"] == origin) &
            (synthesis_df_scaled["horizon"].isin(["impact", "1yr"]))
        )

        block_data_scaled[country][origin] = (
            synthesis_df_scaled.loc[mask, "significant_68"].values
        )

observed_counts_scaled = {
    origin: sum(
        block_data_scaled[country][origin].sum()
        for country in TARGET_COUNTRIES
    )
    for origin in origins
}

null_counts_scaled = []

for assignment in product(origins, repeat=len(TARGET_COUNTRIES)):
    total = sum(
        block_data_scaled[country][origin].sum()
        for country, origin in zip(TARGET_COUNTRIES, assignment)
    )
    null_counts_scaled.append(total)

null_counts_scaled = np.asarray(null_counts_scaled)

permutation_scaled_rows = []

for origin in origins:
    observed = observed_counts_scaled[origin]
    p_value = np.mean(null_counts_scaled >= observed)

    permutation_scaled_rows.append({
        "gpr_origin": origin,
        "count_S0": observed_counts_S0[origin],
        "count_scaled": observed,
        "p_value_scaled": p_value,
    })

permutation_scaled_df = pd.DataFrame(permutation_scaled_rows)

print("\nOrigin heterogeneity under scaled-S, alpha=0:")
print(f"Exact assignments: {len(null_counts_scaled)}")
display(permutation_scaled_df.round(6))

Total comparisons: 240
Sign agreement S=0 vs scaled-S: 100.0%
Significance changes: 2 of 240

68% band-width ratio (scaled-S / S=0):
Mean:   1.008
Q25:    1.005
Median: 1.007
Q75:    1.010

Cases with a change in 68% significance:


,country,gpr_origin,response_var,horizon,median_S0,median_scaled,sig_S0,sig_scaled,width_ratio_scaled_over_S0
84,France,China,dlog_IPI,impact,0.00133,0.00133,True,False,1.02289
223,Spain,MENA,CLIFS_level,1yr,-0.00402,-0.00402,True,False,1.00628



Origin heterogeneity under scaled-S, alpha=0:
Exact assignments: 256


,gpr_origin,count_S0,count_scaled,p_value_scaled
0,Russia,23,23,0.023438
1,China,15,14,0.566406
2,MENA,11,10,0.953125
3,Western,11,11,0.835938


## E. Levels versus differences and cointegration diagnostics

**Design note, not changed silently**: variables entering in levels get a random-walk-consistent own-lag1 prior mean (1), while GPR still gets its estimated AR(1) coefficient -- this is the existing design choice, preserved here (not forced to 0). This is the only block where `own_lag1_means` is passed explicitly to the shared sampler.

In [12]:
# ============================================================
# D0. LOAD PANEL_USD AND BUILD LOG-LEVEL COLUMNS
# Prepare level variables for the levels-vs-differences robustness.
# ============================================================

panel_usd_raw = pd.read_excel(CLEAN_PANEL_PATH)
panel_usd_raw["quarter"] = pd.PeriodIndex(panel_usd_raw["quarter"].astype(str), freq="Q")

panel_usd_raw["log_Brent"] = np.log(panel_usd_raw["brent_usd"])
panel_usd_raw["log_Gas"] = np.log(panel_usd_raw["gas_usd"])
panel_usd_raw["log_HICP"] = np.log(panel_usd_raw["hicp"])
panel_usd_raw["log_IPI"] = np.log(panel_usd_raw["ipi"])

level_cols = ["log_Brent", "log_Gas", "log_HICP", "log_IPI"]

# Safe re-execution: avoid _x/_y columns if D0 has already been run.
panel = panel.drop(columns=[c for c in level_cols if c in panel.columns])

panel = panel.merge(
    panel_usd_raw[["quarter", "country"] + level_cols],
    on=["quarter", "country"],
    how="left"
)

print("Log-level columns merged into panel.")

Log-level columns merged into panel.


In [13]:
# ============================================================
# D1. LEVELS VS. DIFFERENCES -- ALL 16 SYSTEMS
# Baseline:
# [GPR, dlog Brent, dlog Gas, dlog HICP, dlog IPI, CLIFS]
#
# Alternative:
# [GPR, log Brent, log Gas, log HICP, log IPI, CLIFS]
#
# Only the transformation of Brent, Gas, HICP and IPI changes.
# GPR and CLIFS retain their baseline treatment.
# Frozen COVID theta, S=0, alpha=0 and all other BVAR settings
# are kept unchanged.
# ============================================================

LEVEL_COLS = {
    "dlog_Brent": "log_Brent",
    "dlog_Gas": "log_Gas",
    "dlog_HICP": "log_HICP",
    "dlog_IPI": "log_IPI",
    "CLIFS_level": "CLIFS_level",
}


def bvar_minnesota_gibbs_levels_final(Y, quarters, p, gpr_idx, theta,
                                       reps=REPS, burn=BURN, seed=SEED):
    """
    Levels-specification wrapper around the shared sampler.

    Prior own-lag-1 means:
    - GPR: estimated AR(1), enforced inside the shared sampler.
    - Brent, Gas, HICP and IPI in log-levels: 1.
    - CLIFS: 0, unchanged from the baseline specification.

    The frozen COVID volatility path from NB04 is retained to isolate
    the effect of changing the variable transformations.
    """
    n = Y.shape[1]

    own_lag1_means = np.ones(n)
    own_lag1_means[gpr_idx] = 0.0   # overwritten defensively by phi_GPR
    own_lag1_means[-1] = 0.0        # CLIFS remains a stationary level variable

    return bvar_minnesota_gibbs_final(
        Y, quarters, p, gpr_idx, theta,
        own_lag1_means=own_lag1_means,
        reps=reps, burn=burn, seed=seed
    )


# ------------------------------------------------------------
# Run all 16 alternative level systems
# ------------------------------------------------------------

bvar_results_levels_full = {}

for country in TARGET_COUNTRIES:
    sub = panel[panel["country"] == country].sort_values("quarter").reset_index(drop=True)

    for gpr_col in GPR_ORIGINS_COLS:
        origin = next(o for o, col in GPR_ORIGIN_COLS.items() if col == gpr_col)
        var_order = [gpr_col] + [LEVEL_COLS[v] for v in CHOLESKY_ORDER_CORE]

        data_full = sub[["quarter"] + var_order].dropna()
        quarters = data_full["quarter"].astype(str).reset_index(drop=True)
        data = data_full[var_order].values

        theta = theta_lookup[(country, origin)]

        beta_draws, sigma_draws, phi_gpr = bvar_minnesota_gibbs_levels_final(
            data, quarters, p=LAG, gpr_idx=0, theta=theta
        )

        Theta_draws = bvar_structural_irfs(
            beta_draws, sigma_draws, n=6, p=LAG, horizon=IRF_HORIZON
        )

        bvar_results_levels_full[(country, origin)] = (Theta_draws, phi_gpr, var_order)

    print(f"{country}: done (levels, all 4 origins).")

print(f"\nLevels version estimated for all {len(bvar_results_levels_full)} systems.")


# ------------------------------------------------------------
# Impact-sign comparison
# At impact, the differenced response is also the first-period
# cumulative level response, so the qualitative sign is comparable.
# ------------------------------------------------------------

levels_diff_comparison = []

for (country, origin), (Theta_lvl, _, var_order_lvl) in bvar_results_levels_full.items():
    Theta_diff, _, var_order_diff = bvar_results[(country, origin)]

    median_diff = np.median(Theta_diff, axis=0)
    median_lvl = np.median(Theta_lvl, axis=0)

    for i, var_name in enumerate(CHOLESKY_ORDER_CORE, start=1):
        val_diff = median_diff[0, i, 0]
        val_lvl = median_lvl[0, i, 0]

        levels_diff_comparison.append({
            "country": country,
            "gpr_origin": origin,
            "response_var": var_name,
            "diff_value": val_diff,
            "levels_value": val_lvl,
            "same_sign": np.sign(val_diff) == np.sign(val_lvl),
        })

levels_diff_df = pd.DataFrame(levels_diff_comparison)

print(f"\nTotal impact comparisons: {len(levels_diff_df)}")
print(
    f"Same sign: {levels_diff_df['same_sign'].sum()} of {len(levels_diff_df)} "
    f"({100 * levels_diff_df['same_sign'].mean():.1f}%)"
)

print("\nSign agreement rate by variable:")
display(levels_diff_df.groupby("response_var")["same_sign"].agg(["sum", "count"]))

print("\nCases where the impact sign differs:")
display(levels_diff_df.loc[~levels_diff_df["same_sign"]].round(5))

Germany: done (levels, all 4 origins).
France: done (levels, all 4 origins).
Italy: done (levels, all 4 origins).
Spain: done (levels, all 4 origins).

Levels version estimated for all 16 systems.

Total impact comparisons: 80
Same sign: 67 of 80 (83.8%)

Sign agreement rate by variable:


,sum,count
response_var,,
CLIFS_level,14,16
dlog_Brent,15,16
dlog_Gas,13,16
dlog_HICP,12,16
dlog_IPI,13,16



Cases where the impact sign differs:


,country,gpr_origin,response_var,diff_value,levels_value,same_sign
1,Germany,Russia,dlog_Gas,-0.02308,0.00096,False
3,Germany,Russia,dlog_IPI,-0.00011,0.00065,False
4,Germany,Russia,CLIFS_level,-0.00209,0.00142,False
6,Germany,China,dlog_Gas,-0.01052,0.00579,False
14,Germany,MENA,CLIFS_level,-0.00200,0.00163,False
35,France,Western,dlog_Brent,0.00133,-0.00350,False
37,France,Western,dlog_HICP,0.00001,-0.00003,False
38,France,Western,dlog_IPI,-0.00017,0.00019,False
46,Italy,China,dlog_Gas,-0.01508,0.00495,False
52,Italy,MENA,dlog_HICP,0.00008,-0.00033,False


In [14]:
# ============================================================
# D2. LEVELS VS. DIFFERENCES -- DIRECT ROBUSTNESS COMPARISON
#
# Baseline:
# [GPR, dlog Brent, dlog Gas, dlog HICP, dlog IPI, CLIFS]
#
# Levels alternative:
# [GPR, log Brent, log Gas, log HICP, log IPI, CLIFS]
#
# The comparison is qualitative: sign and 68% significance status.
# Magnitudes are not interpreted as directly equivalent because the
# transformed variables represent different statistical objects.
# ============================================================

LEVEL_NAME_MAP = {
    "dlog_Brent": "log_Brent",
    "dlog_Gas": "log_Gas",
    "dlog_HICP": "log_HICP",
    "dlog_IPI": "log_IPI",
    "CLIFS_level": "CLIFS_level",
}

levels_direct_rows = []

for key in bvar_results:
    country, origin = key

    Theta_diff, _, var_order_diff = bvar_results[key]
    Theta_lvl, _, var_order_lvl = bvar_results_levels_full[key]

    for diff_var, lvl_var in LEVEL_NAME_MAP.items():
        i_diff = var_order_diff.index(diff_var)
        i_lvl = var_order_lvl.index(lvl_var)

        for h_label, h in HORIZONS_TO_CHECK.items():
            draws_diff = Theta_diff[:, h, i_diff, 0]
            draws_lvl = Theta_lvl[:, h, i_lvl, 0]

            med_diff = np.median(draws_diff)
            med_lvl = np.median(draws_lvl)

            low_diff, high_diff = np.percentile(
                draws_diff, [CI_LOW_68, CI_HIGH_68]
            )
            low_lvl, high_lvl = np.percentile(
                draws_lvl, [CI_LOW_68, CI_HIGH_68]
            )

            sig_diff = (low_diff > 0) or (high_diff < 0)
            sig_lvl = (low_lvl > 0) or (high_lvl < 0)

            levels_direct_rows.append({
                "country": country,
                "gpr_origin": origin,
                "response_var": diff_var,
                "horizon": h_label,
                "baseline_median": med_diff,
                "levels_median": med_lvl,
                "same_sign": np.sign(med_diff) == np.sign(med_lvl),
                "sig_baseline": sig_diff,
                "sig_levels": sig_lvl,
                "sig_change": sig_diff != sig_lvl,
            })

levels_direct_df = pd.DataFrame(levels_direct_rows)

print(f"Total comparisons: {len(levels_direct_df)}")
print(
    f"Sign agreement: {levels_direct_df['same_sign'].sum()} of "
    f"{len(levels_direct_df)} "
    f"({100 * levels_direct_df['same_sign'].mean():.1f}%)"
)
print(
    f"Significance changes: {levels_direct_df['sig_change'].sum()} of "
    f"{len(levels_direct_df)}"
)

print("\nSign agreement by variable:")
display(
    levels_direct_df.groupby("response_var")["same_sign"]
    .agg(["sum", "count"])
)

print("\nSign agreement by horizon:")
display(
    levels_direct_df.groupby("horizon")["same_sign"]
    .agg(["sum", "count"])
)

print("\nCases with a significance change:")
display(
    levels_direct_df.loc[
        levels_direct_df["sig_change"],
        [
            "country", "gpr_origin", "response_var", "horizon",
            "baseline_median", "levels_median",
            "sig_baseline", "sig_levels"
        ]
    ].round(5)
)

Total comparisons: 240
Sign agreement: 178 of 240 (74.2%)
Significance changes: 87 of 240

Sign agreement by variable:


,sum,count
response_var,,
CLIFS_level,36,48
dlog_Brent,41,48
dlog_Gas,28,48
dlog_HICP,39,48
dlog_IPI,34,48



Sign agreement by horizon:


,sum,count
horizon,,
1yr,55,80
2yr,56,80
impact,67,80



Cases with a significance change:


,country,gpr_origin,response_var,horizon,baseline_median,levels_median,sig_baseline,sig_levels
3,Germany,Russia,dlog_Gas,impact,-0.02308,0.00096,True,False
4,Germany,Russia,dlog_Gas,1yr,-0.00954,0.00627,True,False
5,Germany,Russia,dlog_Gas,2yr,-0.00648,-0.02264,True,False
10,Germany,Russia,dlog_IPI,1yr,-0.00120,-0.00232,True,False
11,Germany,Russia,dlog_IPI,2yr,-0.00085,-0.00358,True,False
...,...,...,...,...,...,...,...,...
225,Spain,Western,dlog_Brent,impact,-0.01004,-0.01732,False,True
226,Spain,Western,dlog_Brent,1yr,-0.00065,-0.03485,False,True
227,Spain,Western,dlog_Brent,2yr,-0.00022,-0.02170,False,True
229,Spain,Western,dlog_Gas,1yr,0.00156,-0.02984,False,True


In [15]:
# ============================================================
# D3. ORIGIN HETEROGENEITY -- BASELINE VS. LEVELS
#
# Exact 4^4 = 256 permutation test under each specification.
# Significant responses are counted at impact + 1-year horizons,
# using the 68% credible interval.
#
# Baseline and levels are evaluated in their own specification;
# no cumulative transformation is applied.
# ============================================================

from itertools import product

origins_D = origins.copy()

heterogeneity_direct = levels_direct_df[
    levels_direct_df["horizon"].isin(["impact", "1yr"])
].copy()

# ------------------------------------------------------------
# 1. Store significance blocks by country and origin
# ------------------------------------------------------------

block_baseline = {}
block_levels = {}

for country in TARGET_COUNTRIES:
    block_baseline[country] = {}
    block_levels[country] = {}

    for origin in origins_D:
        mask = (
            (heterogeneity_direct["country"] == country) &
            (heterogeneity_direct["gpr_origin"] == origin)
        )

        block_baseline[country][origin] = (
            heterogeneity_direct.loc[mask, "sig_baseline"].values
        )

        block_levels[country][origin] = (
            heterogeneity_direct.loc[mask, "sig_levels"].values
        )


# ------------------------------------------------------------
# 2. Observed significant-response counts
# ------------------------------------------------------------

observed_baseline = {
    origin: sum(
        block_baseline[country][origin].sum()
        for country in TARGET_COUNTRIES
    )
    for origin in origins_D
}

observed_levels = {
    origin: sum(
        block_levels[country][origin].sum()
        for country in TARGET_COUNTRIES
    )
    for origin in origins_D
}

print("\nSignificant responses by origin, impact + 1yr:")
print("Baseline:", observed_baseline)
print("Levels:  ", observed_levels)


# ------------------------------------------------------------
# 3. Exact permutation distribution -- baseline
# ------------------------------------------------------------

null_baseline = []

for assignment in product(origins_D, repeat=len(TARGET_COUNTRIES)):
    total = sum(
        block_baseline[country][origin].sum()
        for country, origin in zip(TARGET_COUNTRIES, assignment)
    )
    null_baseline.append(total)

null_baseline = np.asarray(null_baseline)


# ------------------------------------------------------------
# 4. Exact permutation distribution -- levels
# ------------------------------------------------------------

null_levels = []

for assignment in product(origins_D, repeat=len(TARGET_COUNTRIES)):
    total = sum(
        block_levels[country][origin].sum()
        for country, origin in zip(TARGET_COUNTRIES, assignment)
    )
    null_levels.append(total)

null_levels = np.asarray(null_levels)


# ------------------------------------------------------------
# 5. Compare heterogeneity results
# ------------------------------------------------------------

heterogeneity_levels_rows = []

for origin in origins_D:
    count_base = observed_baseline[origin]
    count_lvl = observed_levels[origin]

    p_base = np.mean(null_baseline >= count_base)
    p_lvl = np.mean(null_levels >= count_lvl)

    heterogeneity_levels_rows.append({
        "gpr_origin": origin,
        "count_baseline": count_base,
        "p_value_baseline": p_base,
        "count_levels": count_lvl,
        "p_value_levels": p_lvl,
    })

heterogeneity_levels_df = pd.DataFrame(heterogeneity_levels_rows)

print("\nOrigin heterogeneity: baseline vs levels")
print(f"Exact assignments per specification: {len(null_levels)}")

display(heterogeneity_levels_df.round(6))


Significant responses by origin, impact + 1yr:
Baseline: {'Russia': np.int64(23), 'China': np.int64(15), 'MENA': np.int64(11), 'Western': np.int64(11)}
Levels:   {'Russia': np.int64(16), 'China': np.int64(14), 'MENA': np.int64(21), 'Western': np.int64(18)}

Origin heterogeneity: baseline vs levels
Exact assignments per specification: 256


,gpr_origin,count_baseline,p_value_baseline,count_levels,p_value_levels
0,Russia,23,0.027344,16,0.785156
1,China,15,0.542969,14,0.917969
2,MENA,11,0.906250,21,0.101562
3,Western,11,0.906250,18,0.445312


In [16]:
# ============================================================
# D3B. LEVELS SPECIFICATION — TABLES FOR DEFENCE APPENDIX
#
# Uses levels_direct_df created in D2.
# Reports only responses whose 68% credible interval excludes zero.
# ============================================================

origin_order = ["Russia", "China", "MENA", "Western"]

country_order = ["France", "Germany", "Italy", "Spain"]

country_abbr = {
    "France": "FR",
    "Germany": "DE",
    "Italy": "IT",
    "Spain": "ES",
}

horizon_order = [
    ("impact", "Impact"),
    ("1yr", "One year"),
    ("2yr", "Two years"),
]

variable_labels = {
    "dlog_Brent": "Brent",
    "dlog_Gas": "Gas",
    "dlog_HICP": "HICP",
    "dlog_IPI": "IPI",
    "CLIFS_level": "CLIFS",
}


# ------------------------------------------------------------
# 1. Validate against the final TFM
# ------------------------------------------------------------

validation_sample = levels_direct_df[
    levels_direct_df["horizon"].isin(["impact", "1yr"])
    & levels_direct_df["sig_levels"]
]

actual_counts = (
    validation_sample
    .groupby("gpr_origin")
    .size()
    .reindex(origin_order, fill_value=0)
    .astype(int)
    .to_dict()
)

expected_counts = {
    "Russia": 16,
    "China": 14,
    "MENA": 21,
    "Western": 18,
}

print("Observed levels counts:", actual_counts)

assert actual_counts == expected_counts, (
    "These results do not match the final TFM. "
    f"Expected {expected_counts}, but obtained {actual_counts}."
)

print("Validation passed: results match the final TFM.\n")


# ------------------------------------------------------------
# 2. Create the content of each table cell
# ------------------------------------------------------------

def credible_levels_cell(variable, origin, horizon):

    subset = levels_direct_df[
        (levels_direct_df["response_var"] == variable)
        & (levels_direct_df["gpr_origin"] == origin)
        & (levels_direct_df["horizon"] == horizon)
        & (levels_direct_df["sig_levels"])
    ]

    if subset.empty:
        return "--"

    entries = []

    for country in country_order:

        country_row = subset[subset["country"] == country]

        if len(country_row) > 1:
            raise ValueError(
                f"Duplicate result for {country}, {origin}, "
                f"{variable}, {horizon}."
            )

        if len(country_row) == 1:
            median = country_row.iloc[0]["levels_median"]
            sign = "+" if median > 0 else "-"
            entries.append(f"{country_abbr[country]}{sign}")

    return ", ".join(entries) if entries else "--"


# ------------------------------------------------------------
# 3. Construct the tables as DataFrames
# ------------------------------------------------------------

def build_levels_table(variables):

    rows = []

    for variable in variables:

        for origin_index, origin in enumerate(origin_order):

            row = {
                "Response": (
                    variable_labels[variable]
                    if origin_index == 0
                    else ""
                ),
                "GPR origin": origin,
            }

            for horizon, horizon_label in horizon_order:
                row[horizon_label] = credible_levels_cell(
                    variable,
                    origin,
                    horizon,
                )

            rows.append(row)

    return pd.DataFrame(rows)


energy_levels_table = build_levels_table([
    "dlog_Brent",
    "dlog_Gas",
])

macro_levels_table = build_levels_table([
    "dlog_HICP",
    "dlog_IPI",
    "CLIFS_level",
])


print("ENERGY-PRICE RESPONSES — LEVELS SPECIFICATION")
display(energy_levels_table)

print("\nMACRO-FINANCIAL RESPONSES — LEVELS SPECIFICATION")
display(macro_levels_table)


# ------------------------------------------------------------
# 4. Print complete Beamer frames
# ------------------------------------------------------------

def print_beamer_frame(title, table):

    print(r"\begin{frame}{" + title + "}")
    print()
    print(r"\centering")
    print(r"\scriptsize")
    print(r"\setlength{\tabcolsep}{2.8pt}")
    print(r"\renewcommand{\arraystretch}{1.05}")
    print()
    print(r"\begin{tabular}{llccc}")
    print(r"\toprule")
    print(
        r"\textbf{Response} & \textbf{GPR origin} "
        r"& \textbf{Impact} & \textbf{One year} "
        r"& \textbf{Two years} \\"
    )
    print(r"\midrule")

    for index, row in table.iterrows():

        if index > 0 and row["Response"] != "":
            print(r"\midrule")

        print(
            f'{row["Response"]} & {row["GPR origin"]} '
            f'& {row["Impact"]} '
            f'& {row["One year"]} '
            f'& {row["Two years"]} \\\\'
        )

    print(r"\bottomrule")
    print(r"\end{tabular}")
    print()
    print(r"\vfill")
    print(r"{\scriptsize")
    print(
        r"\textit{Notes:} Only responses whose 68\% posterior "
        r"credible interval excludes zero are reported. "
        r"A $+$ ($-$) indicates a positive (negative) "
        r"posterior-median response. Brent, Gas, HICP and IPI "
        r"enter in log-levels, while CLIFS enters in levels."
    )
    print(r"}")
    print()
    print(r"\end{frame}")
    print("\n")


print_beamer_frame(
    "Energy-Price Responses: Levels Robustness",
    energy_levels_table,
)

print_beamer_frame(
    "Macro-Financial Responses: Levels Robustness",
    macro_levels_table,
)

Observed levels counts: {'Russia': 16, 'China': 14, 'MENA': 21, 'Western': 18}
Validation passed: results match the final TFM.

ENERGY-PRICE RESPONSES — LEVELS SPECIFICATION


,Response,GPR origin,Impact,One year,Two years
0,Brent,Russia,"FR+, ES+","FR-, DE-, IT-, ES-","FR-, DE-, IT-, ES-"
1,,China,"DE+, IT+, ES+",--,--
2,,MENA,"FR-, DE-, IT-, ES-","FR-, DE-, IT-, ES-","FR-, DE-, IT-, ES-"
3,,Western,"IT-, ES-","FR-, DE-, IT-, ES-","FR-, DE-, IT-, ES-"
4,Gas,Russia,--,--,FR-
5,,China,--,"DE+, IT+, ES+",--
6,,MENA,IT-,"FR-, DE-, IT-, ES-","FR-, DE-, IT-, ES-"
7,,Western,"FR-, DE-, ES-","IT-, ES-","FR-, DE-, IT-, ES-"



MACRO-FINANCIAL RESPONSES — LEVELS SPECIFICATION


,Response,GPR origin,Impact,One year,Two years
0,HICP,Russia,"DE+, IT+, ES+","DE+, IT+, ES+","DE+, IT+, ES+"
1,,China,"FR+, DE+","FR+, DE+, IT+","FR+, DE+, IT+"
2,,MENA,--,"FR-, IT-","FR-, IT-"
3,,Western,--,DE+,DE+
4,IPI,Russia,IT-,--,--
5,,China,DE+,IT+,--
6,,MENA,DE-,"FR+, ES+","FR+, IT+, ES+"
7,,Western,IT-,ES+,"FR+, ES+"
8,CLIFS,Russia,"FR-, ES-",DE+,"DE+, IT-, ES-"
9,,China,DE-,--,--


\begin{frame}{Energy-Price Responses: Levels Robustness}

\centering
\scriptsize
\setlength{\tabcolsep}{2.8pt}
\renewcommand{\arraystretch}{1.05}

\begin{tabular}{llccc}
\toprule
\textbf{Response} & \textbf{GPR origin} & \textbf{Impact} & \textbf{One year} & \textbf{Two years} \\
\midrule
Brent & Russia & FR+, ES+ & FR-, DE-, IT-, ES- & FR-, DE-, IT-, ES- \\
 & China & DE+, IT+, ES+ & -- & -- \\
 & MENA & FR-, DE-, IT-, ES- & FR-, DE-, IT-, ES- & FR-, DE-, IT-, ES- \\
 & Western & IT-, ES- & FR-, DE-, IT-, ES- & FR-, DE-, IT-, ES- \\
\midrule
Gas & Russia & -- & -- & FR- \\
 & China & -- & DE+, IT+, ES+ & -- \\
 & MENA & IT- & FR-, DE-, IT-, ES- & FR-, DE-, IT-, ES- \\
 & Western & FR-, DE-, ES- & IT-, ES- & FR-, DE-, IT-, ES- \\
\bottomrule
\end{tabular}

\vfill
{\scriptsize
\textit{Notes:} Only responses whose 68\% posterior credible interval excludes zero are reported. A $+$ ($-$) indicates a positive (negative) posterior-median response. Brent, Gas, HICP and IPI enter in log-levels

In [17]:
# ============================================================
# D4. JOHANSEN COINTEGRATION DIAGNOSTICS
#
# D4A: I(1) block
# [log Brent, log Gas, log HICP, log IPI]
#
# This is the primary cointegration test for the levels-vs-
# differences decision because these are the four variables
# transformed to log-differences in the baseline.
#
# D4B: Full 6-variable systems
# [GPR, log Brent, log Gas, log HICP, log IPI, CLIFS]
#
# The full-system exercise is reported as an additional diagnostic.
# Since GPR and CLIFS are treated as stationary level variables,
# the full-system rank cannot be interpreted purely as the number
# of cointegrating relationships among I(1) variables.
#
# Baseline VAR: p = 2
# Johansen/VECM: k_ar_diff = p - 1 = 1
# Deterministic specification: det_order = 0
# Significance level: 5%
# ============================================================

from statsmodels.tsa.vector_ar.vecm import coint_johansen


def johansen_rank_5pct(data, det_order=0, k_ar_diff=LAG - 1):
    """Return Johansen result and sequential trace/max-eigen ranks at 5%."""
    res = coint_johansen(data, det_order=det_order, k_ar_diff=k_ar_diff)
    n = data.shape[1]

    trace_rank = n
    maxeig_rank = n

    for r in range(n):
        if res.lr1[r] <= res.cvt[r, 1]:
            trace_rank = r
            break

    for r in range(n):
        if res.lr2[r] <= res.cvm[r, 1]:
            maxeig_rank = r
            break

    return res, trace_rank, maxeig_rank


# ============================================================
# D4A. JOHANSEN TEST -- I(1) BLOCK
# ============================================================

JOHANSEN_I1_VARS = [
    "log_Brent",
    "log_Gas",
    "log_HICP",
    "log_IPI",
]

johansen_i1_summary = []
johansen_i1_detail = []

for country in TARGET_COUNTRIES:
    sub = (
        panel.loc[
            panel["country"] == country,
            ["quarter"] + JOHANSEN_I1_VARS
        ]
        .sort_values("quarter")
        .dropna()
        .reset_index(drop=True)
    )

    data = sub[JOHANSEN_I1_VARS].values
    res, trace_rank, maxeig_rank = johansen_rank_5pct(data)

    johansen_i1_summary.append({
        "country": country,
        "n_obs": len(sub),
        "trace_rank_5pct": trace_rank,
        "maxeig_rank_5pct": maxeig_rank,
        "same_rank": trace_rank == maxeig_rank,
    })

    for r in range(len(JOHANSEN_I1_VARS)):
        johansen_i1_detail.append({
            "country": country,
            "null_rank_r": r,
            "trace_stat": res.lr1[r],
            "trace_cv_5pct": res.cvt[r, 1],
            "trace_reject_5pct": res.lr1[r] > res.cvt[r, 1],
            "maxeig_stat": res.lr2[r],
            "maxeig_cv_5pct": res.cvm[r, 1],
            "maxeig_reject_5pct": res.lr2[r] > res.cvm[r, 1],
        })

johansen_i1_summary_df = pd.DataFrame(johansen_i1_summary)
johansen_i1_detail_df = pd.DataFrame(johansen_i1_detail)

print("D4A. Johansen ranks for the I(1) block at the 5% level:")
display(johansen_i1_summary_df)

print("\nDetailed I(1)-block Johansen results:")
display(johansen_i1_detail_df.round(3))


# ============================================================
# D4B. JOHANSEN TEST -- FULL 6-VARIABLE SYSTEMS
# ============================================================

johansen_full_summary = []
johansen_full_detail = []

for country in TARGET_COUNTRIES:
    sub_country = (
        panel[panel["country"] == country]
        .sort_values("quarter")
        .reset_index(drop=True)
    )

    for gpr_col in GPR_ORIGINS_COLS:
        origin = next(o for o, col in GPR_ORIGIN_COLS.items() if col == gpr_col)

        var_order = [
            gpr_col,
            "log_Brent",
            "log_Gas",
            "log_HICP",
            "log_IPI",
            "CLIFS_level",
        ]

        sub = sub_country[["quarter"] + var_order].dropna().reset_index(drop=True)
        data = sub[var_order].values

        res, trace_rank, maxeig_rank = johansen_rank_5pct(data)

        johansen_full_summary.append({
            "country": country,
            "gpr_origin": origin,
            "n_obs": len(sub),
            "trace_rank_5pct": trace_rank,
            "maxeig_rank_5pct": maxeig_rank,
            "same_rank": trace_rank == maxeig_rank,
        })

        for r in range(len(var_order)):
            johansen_full_detail.append({
                "country": country,
                "gpr_origin": origin,
                "null_rank_r": r,
                "trace_stat": res.lr1[r],
                "trace_cv_5pct": res.cvt[r, 1],
                "trace_reject_5pct": res.lr1[r] > res.cvt[r, 1],
                "maxeig_stat": res.lr2[r],
                "maxeig_cv_5pct": res.cvm[r, 1],
                "maxeig_reject_5pct": res.lr2[r] > res.cvm[r, 1],
            })

johansen_full_summary_df = pd.DataFrame(johansen_full_summary)
johansen_full_detail_df = pd.DataFrame(johansen_full_detail)

print("\nD4B. Johansen ranks for the full 6-variable systems at the 5% level:")
display(johansen_full_summary_df)

print("\nFull-system rank distribution:")
display(
    johansen_full_summary_df
    .groupby(["trace_rank_5pct", "maxeig_rank_5pct"])
    .size()
    .reset_index(name="n_systems")
)

print("\nDetailed full-system Johansen results:")
display(johansen_full_detail_df.round(3))


# ============================================================
# D4C. FINAL DIAGNOSTIC SUMMARY
# ============================================================

print("\nFinal cointegration diagnostic:")
print(
    "I(1) block -- trace ranks:",
    johansen_i1_summary_df.set_index("country")["trace_rank_5pct"].to_dict()
)
print(
    "I(1) block -- max-eigen ranks:",
    johansen_i1_summary_df.set_index("country")["maxeig_rank_5pct"].to_dict()
)

print(
    f"\nFull systems with trace/max-eigen rank agreement: "
    f"{johansen_full_summary_df['same_rank'].sum()} of "
    f"{len(johansen_full_summary_df)}"
)

D4A. Johansen ranks for the I(1) block at the 5% level:


,country,n_obs,trace_rank_5pct,maxeig_rank_5pct,same_rank
0,Germany,115,1,1,True
1,France,115,1,1,True
2,Italy,115,1,1,True
3,Spain,115,1,1,True



Detailed I(1)-block Johansen results:


,country,null_rank_r,trace_stat,trace_cv_5pct,trace_reject_5pct,maxeig_stat,maxeig_cv_5pct,maxeig_reject_5pct
0,Germany,0,63.863,47.854,True,47.925,27.586,True
1,Germany,1,15.938,29.796,False,8.482,21.131,False
2,Germany,2,7.456,15.494,False,5.460,14.264,False
3,Germany,3,1.995,3.842,False,1.995,3.842,False
4,France,0,51.979,47.854,True,31.484,27.586,True
5,France,1,20.495,29.796,False,15.981,21.131,False
6,France,2,4.514,15.494,False,4.301,14.264,False
7,France,3,0.213,3.842,False,0.213,3.842,False
8,Italy,0,58.017,47.854,True,36.815,27.586,True
9,Italy,1,21.203,29.796,False,17.634,21.131,False



D4B. Johansen ranks for the full 6-variable systems at the 5% level:


...


Full-system rank distribution:


,trace_rank_5pct,maxeig_rank_5pct,n_systems
0,1,0,1
1,1,1,3
2,2,0,2
3,2,1,4
4,2,2,6



Detailed full-system Johansen results:


,country,gpr_origin,null_rank_r,trace_stat,trace_cv_5pct,trace_reject_5pct,maxeig_stat,maxeig_cv_5pct,maxeig_reject_5pct
0,Germany,Russia,0,126.362,95.754,True,50.090,40.076,True
1,Germany,Russia,1,76.272,69.819,True,34.718,33.878,True
2,Germany,Russia,2,41.553,47.854,False,24.654,27.586,False
3,Germany,Russia,3,16.900,29.796,False,9.831,21.131,False
4,Germany,Russia,4,7.069,15.494,False,5.564,14.264,False
...,...,...,...,...,...,...,...,...,...
91,Spain,Western,1,86.724,69.819,True,39.802,33.878,True
92,Spain,Western,2,46.922,47.854,False,24.968,27.586,False
93,Spain,Western,3,21.955,29.796,False,14.820,21.131,False
94,Spain,Western,4,7.134,15.494,False,6.988,14.264,False



Final cointegration diagnostic:
I(1) block -- trace ranks: {'Germany': 1, 'France': 1, 'Italy': 1, 'Spain': 1}
I(1) block -- max-eigen ranks: {'Germany': 1, 'France': 1, 'Italy': 1, 'Spain': 1}

Full systems with trace/max-eigen rank agreement: 9 of 16


## F. Additional Gas sample-sensitivity check

**Design note on E1, preserved as-is (not redesigned)**: the current cell estimates a pre-2022Q3-only subsample and compares it against the FULL-sample estimate -- it does not construct a separate post-2022Q3-only subsample (data after 2022Q3 alone would likely be too short for stable estimation). The question this actually answers is *does dropping everything from 2022Q3 onward change the impact estimate*, not a true pre-vs-post structural-break comparison. This limitation is kept explicit rather than silently reframed.

In [18]:
# ============================================================
# E1. GAS-SIGN DIAGNOSTIC -- PRE-2022Q3 SAMPLE
#
# Compares the full-sample baseline with a BVAR estimated using
# observations only up to 2022Q2.
#
# 2022Q3 is used as an approximate marker for the more recent
# European gas-market regime associated with accelerated LNG
# diversification and reduced Russian pipeline flows.
#
# This is a sample-sensitivity diagnostic, not a causal
# pre-vs-post structural-break test.
#
# All production BVAR settings are retained. The frozen COVID
# theta from NB04 is also retained; prior scales and the GPR
# AR(1) prior mean are recalculated from the truncated sample
# through the shared sampler.
# ============================================================

SPLIT_DATE = "2022Q3"

gas_split_rows = []

for country in TARGET_COUNTRIES:
    sub = (
        panel[panel["country"] == country]
        .sort_values("quarter")
        .reset_index(drop=True)
    )

    sub_pre = sub[sub["quarter"].astype(str) < SPLIT_DATE].reset_index(drop=True)

    for gpr_col in GPR_ORIGINS_COLS:
        origin = next(o for o, col in GPR_ORIGIN_COLS.items() if col == gpr_col)
        var_order = [gpr_col] + CHOLESKY_ORDER_CORE

        data_pre_full = sub_pre[["quarter"] + var_order].dropna()
        quarters_pre = data_pre_full["quarter"].astype(str).reset_index(drop=True)
        data_pre = data_pre_full[var_order].values

        theta = theta_lookup[(country, origin)]

        beta_draws, sigma_draws, phi_gpr = bvar_minnesota_gibbs_final(
            data_pre, quarters_pre, p=LAG, gpr_idx=0, theta=theta
        )

        Theta_pre = bvar_structural_irfs(
            beta_draws, sigma_draws, n=6, p=LAG, horizon=IRF_HORIZON
        )

        Theta_full, _, var_order_full = bvar_results[(country, origin)]

        gas_idx_pre = var_order.index("dlog_Gas")
        gas_idx_full = var_order_full.index("dlog_Gas")

        draws_pre = Theta_pre[:, 0, gas_idx_pre, 0]
        draws_full = Theta_full[:, 0, gas_idx_full, 0]

        med_pre = np.median(draws_pre)
        med_full = np.median(draws_full)

        low_pre, high_pre = np.percentile(
            draws_pre, [CI_LOW_68, CI_HIGH_68]
        )
        low_full, high_full = np.percentile(
            draws_full, [CI_LOW_68, CI_HIGH_68]
        )

        sig_pre = (low_pre > 0) or (high_pre < 0)
        sig_full = (low_full > 0) or (high_full < 0)

        gas_split_rows.append({
            "country": country,
            "gpr_origin": origin,
            "gas_full": med_full,
            "gas_pre_2022Q3": med_pre,
            "negative_full": med_full < 0,
            "negative_pre": med_pre < 0,
            "same_sign": np.sign(med_full) == np.sign(med_pre),
            "sig_full": sig_full,
            "sig_pre": sig_pre,
            "sig_change": sig_full != sig_pre,
        })

    print(f"{country}: done.")

gas_split_df = pd.DataFrame(gas_split_rows)

print("\nGas impact response: full sample vs pre-2022Q3 sample")
display(gas_split_df.round(5))

print(
    f"\nNegative in pre-2022Q3 sample: "
    f"{gas_split_df['negative_pre'].sum()} of {len(gas_split_df)}"
)

print(
    f"Same impact sign as full sample: "
    f"{gas_split_df['same_sign'].sum()} of {len(gas_split_df)}"
)

print(
    f"Significance changes: "
    f"{gas_split_df['sig_change'].sum()} of {len(gas_split_df)}"
)

print(
    "\nDiagnostic only: the exercise tests whether the negative Gas "
    "impact response depends on observations from 2022Q3 onward; "
    "it does not identify LNG diversification as a causal mechanism."
)

Germany: done.
France: done.
Italy: done.
Spain: done.

Gas impact response: full sample vs pre-2022Q3 sample


,country,gpr_origin,gas_full,gas_pre_2022Q3,negative_full,negative_pre,same_sign,sig_full,sig_pre,sig_change
0,Germany,Russia,-0.02308,-0.01871,True,True,True,True,True,False
1,Germany,China,-0.01052,-0.00995,True,True,True,False,False,False
2,Germany,MENA,-0.00975,-0.02165,True,True,True,False,True,True
3,Germany,Western,-0.03240,-0.03150,True,True,True,True,True,False
4,France,Russia,-0.02596,-0.02030,True,True,True,True,True,False
5,France,China,-0.00976,-0.00986,True,True,True,False,False,False
6,France,MENA,-0.00006,-0.01108,True,True,True,False,False,False
7,France,Western,-0.02577,-0.02819,True,True,True,True,True,False
8,Italy,Russia,-0.03100,-0.02372,True,True,True,True,True,False
9,Italy,China,-0.01508,-0.01356,True,True,True,False,True,True



Negative in pre-2022Q3 sample: 16 of 16
Same impact sign as full sample: 16 of 16
Significance changes: 3 of 16

Diagnostic only: the exercise tests whether the negative Gas impact response depends on observations from 2022Q3 onward; it does not identify LNG diversification as a causal mechanism.


## G. Robustness synthesis and export


In [19]:
# ============================================================
# G. ROBUSTNESS SYNTHESIS AND EXPORT
# ============================================================
summary_rows = []

if "reduced_comparison_df" in globals():
    summary_rows.append({
        "specification": "Reduced system",
        "comparisons": len(reduced_comparison_df),
        "sign_agreement_pct": 100 * reduced_comparison_df["same_sign"].mean(),
        "detectability_changes": int(reduced_comparison_df["sig_change"].sum()),
    })

if "scaled_prior_df" in globals():
    summary_rows.append({
        "specification": "Scale-aware covariance prior",
        "comparisons": len(scaled_prior_df),
        "sign_agreement_pct": 100 * scaled_prior_df["same_sign"].mean(),
        "detectability_changes": int(scaled_prior_df["sig_change"].sum()),
    })

if "sigma_prior_df" in globals():
    summary_rows.append({
        "specification": "Identity covariance prior",
        "comparisons": len(sigma_prior_df),
        "sign_agreement_pct": 100 * sigma_prior_df["same_sign"].mean(),
        "detectability_changes": int(sigma_prior_df["sig_change"].sum()),
    })

if "levels_direct_df" in globals():
    summary_rows.append({
        "specification": "Levels specification",
        "comparisons": len(levels_direct_df),
        "sign_agreement_pct": 100 * levels_direct_df["same_sign"].mean(),
        "detectability_changes": int(levels_direct_df["sig_change"].sum()),
    })

robustness_summary = pd.DataFrame(summary_rows)
ROBUSTNESS_SUMMARY_PATH = OUTPUT_DIR / "robustness_summary.csv"
robustness_summary.to_csv(ROBUSTNESS_SUMMARY_PATH, index=False)
display(robustness_summary.round(3))

# Gas-specific diagnostics
gas_tables = []
for name, obj in [
    ("brent_gas_inversion", globals().get("gas_sign_df")),
    ("energy_first", globals().get("gpr_energy_ordering_df")),
    ("exclude_2021Q4_2022Q4", globals().get("gas_crisis_df")),
    ("pre_2022Q3", globals().get("gas_split_df")),
]:
    if isinstance(obj, pd.DataFrame):
        tmp = obj.copy()
        tmp.insert(0, "robustness_check", name)
        gas_tables.append(tmp)

if gas_tables:
    gas_robustness = pd.concat(gas_tables, ignore_index=True, sort=False)
    gas_robustness.to_csv(OUTPUT_DIR / "gas_robustness_summary.csv", index=False)

for name, obj in [
    ("reduced_system_comparison.csv", globals().get("reduced_comparison_df")),
    ("sigma_prior_identity_comparison.csv", globals().get("sigma_prior_df")),
    ("sigma_prior_scaled_comparison.csv", globals().get("scaled_prior_df")),
    ("levels_comparison.csv", globals().get("levels_direct_df")),
    ("levels_heterogeneity.csv", globals().get("heterogeneity_levels_df")),
]:
    if isinstance(obj, pd.DataFrame):
        obj.to_csv(TABLE_DIR / name, index=False)

print(f"Robustness outputs saved to: {OUTPUT_DIR}")


,specification,comparisons,sign_agreement_pct,detectability_changes
0,Scale-aware covariance prior,240,100.000,2
1,Identity covariance prior,240,87.917,66
2,Levels specification,240,74.167,87


Robustness outputs saved to: /content/drive/MyDrive/TFM_GitHub/outputs/06_robustness


## Interpretation

The robustness exercises are designed to separate findings that are stable across reasonable modelling alternatives from findings that depend on specification choices. The reduced-system and scale-aware covariance-prior checks assess whether the main macro-financial patterns survive changes in system dimension and prior scaling. The ordering and sample checks focus on the negative short-run Gas response. The levels exercise is intentionally more demanding because it changes the persistence properties of four variables; geographical concentration by GPR origin is therefore evaluated again under that specification rather than assumed to carry over from the baseline.
